# Embeddings and Vector Databases with Pinecone

**Embeddings** are vector representations of text. They are used to represent text in a vector space, where the distance between vectors represents the semantic similarity between the texts.

**Vector Databases** are databases that store vectors and their associated metadata. They are used to store and retrieve embeddings. Vector databases are organized into indexes (also called namespaces) - similar to tables in a relational database.

[Pinecone](https://www.pinecone.io/) is a vector database service that allows you to store and retrieve embeddings. It is a hosted service that allows you to scale your vector database as needed.

There are two main ways to use Pinecone:

1. **Store an embedding** - Store an embedding in a vector database.
    - Embed the text you want to store.
    - Create a document with the embedding and metadata.
    - Store the document in a vector database.
2. **Query a vector database** - Query a vector database for the most similar embeddings to a given query.
    - Embed the query.
    - Query the vector database with the embedded query.
    - Retrieve the most similar embeddings to the query.

### Install Pinecone and OpenAI

In [7]:
%pip install pinecone openai


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Import Libraries

In [4]:
import os
import uuid
from datetime import datetime, timezone
from pinecone import Pinecone
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

True

### Define Environment variables

In [5]:
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY") # Pinecone API key
PINECONE_INDEX_NAME=os.getenv("PINECONE_INDEX_NAME") # Name of the vector database index
PINECONE_NAMESPACE=os.getenv("PINECONE_NAMESPACE") # Namespace in your index on Pinecone.io
EMBEDDING_MODEL=os.getenv("EMBEDDING_MODEL") # OpenAI embedding model
EMBEDDING_DIMENSIONS=os.getenv("EMBEDDING_DIMENSIONS") # Number of dimensions in the embedding

### Initialize Pinecone and OpenAI

In [6]:
# Initialize Pinecone for vector database
pc = Pinecone(PINECONE_API_KEY)
# Initialize the vector database index
index = pc.Index(PINECONE_INDEX_NAME)
# Initialize OpenAI for embeddings 
client = OpenAI()

/Users/nmirabets/GitHub/memory-ai-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Store an embedding
-----------

In [10]:
string_to_store = 'I like the color blue.'

### Embed the string

In [8]:
# OpenAI embeddings
def get_embeddings(string_to_embed):
    response = client.embeddings.create(
        input=string_to_embed,
        model=EMBEDDING_MODEL,
        dimensions=int(EMBEDDING_DIMENSIONS)
    )
    return response.data[0].embedding

In [11]:
vector = get_embeddings(string_to_store)

In [12]:
print(f"Vector representation of {string_to_store}: \n", vector)

Vector representation of I like the color blue.: 
 [0.0282745361328125, -0.024444580078125, -0.054443359375, 0.04052734375, 0.0106048583984375, -0.027374267578125, 0.0102386474609375, -0.011932373046875, -0.066650390625, -0.0203399658203125, 0.0146331787109375, 0.022613525390625, -0.01546478271484375, 0.0272674560546875, 0.034912109375, 0.0292816162109375, -0.0267181396484375, -0.04595947265625, 0.022613525390625, -0.013336181640625, -0.0078277587890625, -0.027496337890625, -0.0168304443359375, 0.04266357421875, 0.0102081298828125, -0.018218994140625, 0.0283355712890625, 0.005840301513671875, -0.002346038818359375, -0.0243682861328125, -0.00018596649169921875, -0.01438140869140625, -0.0633544921875, -0.05877685546875, 0.0245361328125, -0.047454833984375, 0.038055419921875, 0.03985595703125, -0.0089263916015625, 0.0025730133056640625, 0.005863189697265625, 0.0136260986328125, 0.06024169921875, 0.0015096664428710938, -0.0145416259765625, -0.002170562744140625, -0.04901123046875, -0.02433

### Define the vector metadata to store in the vector database

In [13]:
user_id = "1234"
current_time = datetime.now(tz=timezone.utc)

### Build the vector document to be stored

In [14]:
# Build document dictionary
documents = [
    {
        "id": str(uuid.uuid4()),
        "values": vector,
        "metadata": {
            "user_id": user_id,
            "timestamp": str(current_time),
            "payload": string_to_store,
        },
    }
]


### Store the vector document in the vector database

In [15]:
index.upsert(
    namespace=PINECONE_NAMESPACE,
    vectors=documents,
)

{'upserted_count': 1}

## 2. Query a vector database
-----------

In [16]:
query_string = "What is my favorite color?"
user_id = "1234"
top_k = 1 # This is the number of most similar embeddings to return

### Embed the query

In [17]:
vector = get_embeddings(query_string)

### Query the vector database for similar top_k embeddings + filters

In [18]:
response = index.query(
    vector=vector,
    filter={
        "user_id": {"$eq": user_id},
    },
    namespace=PINECONE_NAMESPACE,
    include_metadata=True,
    top_k=top_k,
)

In [19]:
response

{'matches': [{'id': '59b65de7-2162-4acd-bede-da40636ff5f9',
              'metadata': {'payload': 'I like the color blue.',
                           'timestamp': '2026-08-24 10:08:47.544266+00:00',
                           'user_id': '1234'},
              'score': 0.569750786,
              'values': []}],
 'namespace': 'https://data-memory-2xdin0a.svc.aped-4627-b74a.pinecone.io',
 'usage': {'read_units': 1}}

### Build the memories list

In [16]:
memories = []
if matches := response.get("matches"):
    memories = [m["metadata"]["payload"] for m in matches]
memories

['I like the color blue.']

In [20]:
from agent.tools import save_memory

memories = ['My favorite fruit is mango.', 'I like spaguetti.']

save_memory(memories)

'2 memories saved successfully'

In [23]:
from agent.tools import load_memories

prompt = "What is my favorite color?"

load_memories(prompt)

['I like the color blue.', 'My favorite fruit is mango.']